# Ultimate GTF Strategy Backtester — Google Colab

Backtests the **GTF supply/demand zone strategy** on NSE equities, entirely in the browser.
No local Python install, no broker account, no API key.

---

### ⚠️ Disclaimer

> *"Investment in securities market are subject to market risks. Read all the related documents carefully before investing."*

This notebook is for **educational and research purposes only**. It is not a recommendation to buy or sell
any financial instrument. Backtest results are based on historical data and **do not guarantee future
performance**. Free market data is unofficial and can contain errors, so verify anything surprising
before acting on it. Automated trading carries significant risk — always test in a paper-trading environment first.

---

### How to use

Run the cells **top to bottom**. Each step is a numbered section:

| Step | What it does |
| --- | --- |
| 1 | Install dependencies |
| 2 | Pull the strategy code from GitHub |
| 3 | Choose your data source |
| 4 | Choose which stocks to test |
| 5 | Set the backtest parameters |
| 6 | Run the backtest |
| 7 | Download the Excel report and view the charts |

**You will need: nothing.** No broker account, no API key, no token. The default data source is
Yahoo Finance, which is free and needs no login.

> **Note on runtimes:** Colab wipes the VM when the runtime disconnects. If that happens, just
> re-run the notebook from Step 1.

---
## Step 1 — Install dependencies

`pandas` and `numpy` ship with Colab; this adds Yahoo Finance, the Excel writer and the timezone
data. Takes about 20 seconds.

In [ ]:
#@title Install packages { display-mode: "form" }
!pip install -q yfinance openpyxl pytz tabulate

import pandas as pd, numpy as np, yfinance as yf
print("pandas", pd.__version__, "| numpy", np.__version__, "| yfinance", yf.__version__)
print("Ready \u2713  (no broker account or API key needed)")

---
## Step 2 — Get the strategy code

Clones the repo into `/content/` and imports `gtf_strategy_automation` as `gtf`. Re-running this cell
pulls the latest changes.

The strategy logic lives in the repo, not in this notebook — so any fix pushed to GitHub shows up here
on the next run.

In [ ]:
#@title Clone the repository and import the strategy { display-mode: "form" }
import os, sys, subprocess, importlib

REPO_URL = "https://github.com/ersid111/Stocks.git"  #@param {type:"string"}
BRANCH   = "claude/strategy-document-review-kxjk0s"  #@param {type:"string"}

CLONE_DIR = "/content/Stocks"
WORK_DIR  = os.path.join(CLONE_DIR, "gtf_strategy")

if os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    print("Repo already present \u2014 pulling latest...")
    subprocess.run(["git", "-C", CLONE_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "pull", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, CLONE_DIR],
                   check=True)

# Work inside the strategy folder so the token file and reports land together
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

import gtf_strategy_automation as gtf
importlib.reload(gtf)   # pick up changes if the cell is re-run

print(f"\nWorking directory: {os.getcwd()}")
print(f"Strategy module loaded \u2713  (HTF={gtf.HTF_INTERVAL}, LTF={gtf.LTF_INTERVAL})")

---
## Step 3 — Choose your data source

**`yfinance` (default) needs no account, no API key and no token** — leave it as is and move on.

The one thing to know: **Yahoo only serves about 60 days of `15m` history** (and ~730 days of
`60m`). The default entry timeframe is `15m`, so your date range in Step 5 must be recent.
Ask for something older and you'll get a printed warning and no data, not a crash.

| Source | Account | Notes |
| --- | --- | --- |
| `yfinance` | none | Yahoo Finance. NSE symbols get `.NS` added automatically. |
| `csv` | none | Your own files, uploaded to the Colab session. No network. |
| `fyers` | Fyers + token | Only if you specifically want it — see the repo README. |

In [ ]:
#@title Select the data source { display-mode: "form" }
DATA_PROVIDER = "yfinance"  #@param ["yfinance", "csv", "fyers"]

gtf.DATA_PROVIDER = DATA_PROVIDER

if DATA_PROVIDER == "yfinance":
    print("Source: Yahoo Finance — nothing to set up ✓")
    print("Remember: ~60 days of 15m history, ~730 days of 60m.")

elif DATA_PROVIDER == "csv":
    gtf.CSV_DIR = os.path.join(WORK_DIR, "csv_data")
    os.makedirs(gtf.CSV_DIR, exist_ok=True)
    print(f"Source: local CSV files from {gtf.CSV_DIR}")
    print("Name them <TICKER>_<interval>.csv, e.g. RELIANCE_15m.csv and RELIANCE_60m.csv,")
    print("each with a date column plus Open,High,Low,Close,Volume.")
    try:
        from google.colab import files
        print("")
        print("Upload them now (or skip and use the sidebar file browser):")
        for name, blob in files.upload().items():
            with open(os.path.join(gtf.CSV_DIR, name), "wb") as fh:
                fh.write(blob)
            print(f"  saved {name}")
    except Exception as e:
        print(f"(Upload widget unavailable: {e})")

elif DATA_PROVIDER == "fyers":
    print("The Fyers path needs an access token. See gtf_strategy/README.md —")
    print("install fyers-apiv3, set CLIENT_ID/SECRET_KEY, and write the token to")
    print(f"{os.path.join(WORK_DIR, 'fyers_access_token.txt')}, then call gtf.initialize_fyers().")

---
## Step 4 — Choose your stocks

Plain **NSE symbols**, comma- or newline-separated — the provider adds the `.NS` suffix for you.
Indices work too, written the Yahoo way (`^NSEI` for Nifty 50). Lines starting with `#` are ignored.

In [ ]:
#@title Set the stock list { display-mode: "form" }
SYMBOLS = "HDFCBANK, RELIANCE, SBIN, INFY, TCS"  #@param {type:"string"}

tickers = [s.strip().upper() for s in SYMBOLS.replace("\n", ",").split(",")
           if s.strip() and not s.strip().startswith("#")]

with open(os.path.join(WORK_DIR, "stock_list.txt"), "w") as f:
    f.write("\n".join(tickers) + "\n")

print(f"{len(tickers)} symbols queued:")
print(", ".join(tickers))

---
## Step 5 — Backtest parameters

These overwrite the defaults in the strategy module for this session only — the file on disk is untouched.

**Watch the warmup.** The backtest loop only starts after `EMA_TP_PERIOD + ATR_LOOKBACK + 2` bars
(216 by default). At 15m that's roughly 9 trading days of data consumed before the first possible trade,
so give yourself a date range comfortably longer than that or you'll get zero trades.

**Mind the Yahoo history limit** if you're on the default source: `15m` and `30m` only go back about
60 days, `60m` about 730 days, `1d` for years. A range older than that returns a warning and no data.
The dates below default to a window ending today so this works out of the box.

In [ ]:
#@title Set parameters { display-mode: "form" }
#@markdown ### Date range
#@markdown Leave blank to use the last 55 days, which stays inside Yahoo's 15m limit.
START_DATE = ""  #@param {type:"string"}
END_DATE   = ""  #@param {type:"string"}

#@markdown ### Timeframes
HTF_INTERVAL = "60m"  #@param ["5m", "15m", "30m", "60m", "1d"]
LTF_INTERVAL = "15m"  #@param ["5m", "15m", "30m", "60m", "1d"]

#@markdown ### Capital and risk
INITIAL_CAPITAL = 100000  #@param {type:"number"}
RISK_PER_TRADE  = 1000    #@param {type:"number"}
MAX_R_MULTIPLE  = 3.0     #@param {type:"number"}

#@markdown ### Zone and filter settings
MAX_BASE_CANDLES            = 3     #@param {type:"integer"}
BASE_CANDLE_RATIO           = 0.33  #@param {type:"number"}
MIN_TRADE_SCORE             = 4.5   #@param {type:"number"}
MIN_WICK_TO_RANGE_RATIO     = 0.4   #@param {type:"number"}
MAX_ZONE_WIDTH_ATR_MULTIPLE = 1.5   #@param {type:"number"}
ATR_LOOKBACK                = 14    #@param {type:"integer"}
EMA_TP_PERIOD               = 200   #@param {type:"integer"}

#@markdown ### Pacing
#@markdown Seconds to wait between symbols, so the data source doesn't throttle you.
API_DELAY_SECONDS = 1.0  #@param {type:"number"}

from datetime import date as _date, timedelta as _td

# Default to a recent window that fits inside Yahoo's intraday history limit
if not END_DATE.strip():
    END_DATE = _date.today().isoformat()
if not START_DATE.strip():
    START_DATE = (_date.fromisoformat(END_DATE) - _td(days=55)).isoformat()

gtf.START_DATE      = START_DATE
gtf.END_DATE        = END_DATE
gtf.HTF_INTERVAL    = HTF_INTERVAL
gtf.LTF_INTERVAL    = LTF_INTERVAL
gtf.INITIAL_CAPITAL = float(INITIAL_CAPITAL)
gtf.RISK_PER_TRADE  = float(RISK_PER_TRADE)
gtf.MAX_R_MULTIPLE  = float(MAX_R_MULTIPLE)
gtf.MAX_BASE_CANDLES            = int(MAX_BASE_CANDLES)
gtf.BASE_CANDLE_RATIO           = float(BASE_CANDLE_RATIO)
gtf.MIN_TRADE_SCORE             = float(MIN_TRADE_SCORE)
gtf.MIN_WICK_TO_RANGE_RATIO     = float(MIN_WICK_TO_RANGE_RATIO)
gtf.MAX_ZONE_WIDTH_ATR_MULTIPLE = float(MAX_ZONE_WIDTH_ATR_MULTIPLE)
gtf.ATR_LOOKBACK                = int(ATR_LOOKBACK)
gtf.EMA_TP_PERIOD               = int(EMA_TP_PERIOD)

# The intraday rule set follows the entry timeframe
gtf.IS_INTRADAY_STRATEGY = LTF_INTERVAL in ["5m", "15m", "30m", "60m"]

warmup = gtf.EMA_TP_PERIOD + gtf.ATR_LOOKBACK + 2
print(f"Source        : {gtf.DATA_PROVIDER}")
print(f"Window        : {gtf.START_DATE} \u2192 {gtf.END_DATE} "
      f"({(_date.fromisoformat(gtf.END_DATE) - _date.fromisoformat(gtf.START_DATE)).days} days)")
print(f"Timeframes    : HTF {gtf.HTF_INTERVAL} / LTF {gtf.LTF_INTERVAL}")
print(f"Capital       : Rs {gtf.INITIAL_CAPITAL:,.0f} | risk Rs {gtf.RISK_PER_TRADE:,.0f}/trade "
      f"| target {gtf.MAX_R_MULTIPLE}R")
print(f"Intraday rules: {'ON  (14:45 IST cutoff, no overnight shorts)' if gtf.IS_INTRADAY_STRATEGY else 'OFF (delivery)'}")
print(f"Warmup        : {warmup} bars consumed before the first possible trade")

---
## Step 6 — Run the backtest

Loops over every symbol. Each one needs two fetches (HTF + LTF candles), so a long list takes a while.

The results accumulate in module-level lists, which this cell **clears first** — so you can safely
re-run it after changing parameters without double-counting the previous run's trades.

In [ ]:
#@title Run { display-mode: "form" }
import time as _t

# Clear results from any previous run in this session
gtf.ALL_SUMMARY_RESULTS.clear()
gtf.ALL_COMPLETED_TRADES.clear()

failed = []

for n, ticker in enumerate(tickers, 1):
    print(f"\n[{n}/{len(tickers)}] {ticker}")
    print("-" * 60)
    try:
        result = gtf.run_backtest(ticker, gtf.START_DATE, gtf.END_DATE)
        if result:
            gtf.ALL_SUMMARY_RESULTS.append(result)
        else:
            failed.append(ticker)
    except Exception as e:
        print(f"  ERROR on {ticker}: {type(e).__name__}: {e}")
        failed.append(ticker)

    if n < len(tickers):
        _t.sleep(API_DELAY_SECONDS)

print("\n" + "=" * 60)
print(f"Finished: {len(gtf.ALL_SUMMARY_RESULTS)} symbols completed, {len(failed)} skipped")
if failed:
    print("Skipped (no data or error):", ", ".join(failed))

In [ ]:
#@title Results { display-mode: "form" }
from IPython.display import display

summary_df = pd.DataFrame(gtf.ALL_SUMMARY_RESULTS)
trades_df  = pd.DataFrame(gtf.ALL_COMPLETED_TRADES)

if summary_df.empty:
    print("No completed backtests. Check Step 6's output for API errors, and see the "
          "troubleshooting notes at the bottom of this notebook.")
else:
    summary_df = summary_df.sort_values("Total P&L", ascending=False).reset_index(drop=True)

    total_pnl    = summary_df["Total P&L"].sum()
    total_trades = int(summary_df["Total Trades"].sum())
    wins         = int((trades_df["Status"].eq("TP") |
                        (trades_df["Status"].eq("FORCED_EOD_CLOSE") & trades_df["PnL"].gt(0))).sum()) \
                   if not trades_df.empty else 0
    overall_wr   = (wins / total_trades * 100) if total_trades else 0.0
    avg_r        = trades_df["R_Achieved"].mean() if not trades_df.empty else 0.0

    print("=" * 68)
    print("CONSOLIDATED MULTI-ASSET BACKTEST REPORT")
    print("=" * 68)
    print(f"Portfolio P&L : Rs {total_pnl:,.2f}")
    print(f"Total trades  : {total_trades}   |   Win rate: {overall_wr:.2f}%   |   Avg R: {avg_r:.2f}")
    print(f"Window        : {gtf.START_DATE} \u2192 {gtf.END_DATE}  "
          f"(HTF {gtf.HTF_INTERVAL} / LTF {gtf.LTF_INTERVAL})")
    print("=" * 68)

    display(summary_df.style.format({
        "Win Rate": "{:.2f}%", "Total P&L": "Rs {:,.2f}", "Final Capital": "Rs {:,.2f}",
    }).set_caption("Per-symbol summary"))

    if not trades_df.empty:
        print(f"\nTrade outcomes: {trades_df['Status'].value_counts().to_dict()}")
        display(trades_df.head(25))
    else:
        print("\nNo trades were triggered \u2014 every signal was filtered out. "
              "Try a longer date range or a lower MIN_TRADE_SCORE.")

---
## Step 7 — Export and visualise

The Excel report has two sheets: **Summary_Report** (P&L and win rate per symbol) and
**Detailed_Trades** (every fill, with IST timestamps).

In [ ]:
#@title Download the Excel report { display-mode: "form" }
from datetime import datetime

if summary_df.empty:
    print("Nothing to export yet \u2014 run Step 6 first.")
else:
    report_name = f"Backtest_Report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
    report_path = os.path.join(WORK_DIR, report_name)

    with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
        summary_df.to_excel(writer, sheet_name="Summary_Report", index=False)
        trades_df.to_excel(writer, sheet_name="Detailed_Trades", index=False)

    print(f"Saved: {report_path}")

    try:
        from google.colab import files
        files.download(report_path)
    except Exception as e:
        print(f"(Automatic download unavailable: {e})")
        print("Grab it from the file browser in the left sidebar instead.")

In [ ]:
#@title Charts \u2014 equity curve and P&L by symbol { display-mode: "form" }
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

BLUE, RED = "#2a78d6", "#e34948"      # validated pair: profit / loss
INK, MUTED = "#0b0b0b", "#52514e"

if trades_df.empty:
    print("No trades to plot.")
else:
    plot_df = trades_df.copy()
    plot_df["Exit Date"] = pd.to_datetime(plot_df["Exit Date"])
    plot_df = plot_df.sort_values("Exit Date")
    plot_df["Cumulative P&L"] = plot_df["PnL"].cumsum()

    def _style(ax):
        ax.spines[["top", "right"]].set_visible(False)
        ax.spines[["left", "bottom"]].set_color("#d8d7d2")
        ax.grid(axis="y", color="#eeedea", linewidth=1)
        ax.set_axisbelow(True)
        ax.tick_params(colors=MUTED, labelsize=9, length=0)

    # ── Equity curve: one series, so no legend needed \u2014 the title names it ──
    fig, ax = plt.subplots(figsize=(11, 4.2))
    ax.axhline(0, color="#c9c8c3", linewidth=1)
    ax.plot(plot_df["Exit Date"], plot_df["Cumulative P&L"],
            color=BLUE, linewidth=2, solid_capstyle="round")
    final = plot_df["Cumulative P&L"].iloc[-1]
    ax.scatter([plot_df["Exit Date"].iloc[-1]], [final], color=BLUE, s=42, zorder=3)
    ax.annotate(f"  Rs {final:,.0f}", (plot_df["Exit Date"].iloc[-1], final),
                color=INK, fontsize=10, fontweight="bold", va="center")
    ax.set_title("Cumulative P&L across all trades", color=INK, fontsize=13,
                 fontweight="bold", loc="left", pad=12)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    ax.set_ylabel("Rupees", color=MUTED, fontsize=9)
    _style(ax)
    fig.autofmt_xdate()
    plt.tight_layout(); plt.show()

    # ── P&L by symbol: colour marks the sign, the axis and labels state it ──
    by_sym = summary_df.sort_values("Total P&L")
    fig, ax = plt.subplots(figsize=(11, max(2.4, 0.45 * len(by_sym) + 1.4)))
    colors = [BLUE if v >= 0 else RED for v in by_sym["Total P&L"]]
    ax.barh(by_sym["Ticker"], by_sym["Total P&L"], color=colors, height=0.62)
    ax.axvline(0, color="#c9c8c3", linewidth=1)
    lo, hi = min(0, by_sym["Total P&L"].min()), max(0, by_sym["Total P&L"].max())
    pad = max(hi - lo, 1) * 0.18
    for y, v in zip(by_sym["Ticker"], by_sym["Total P&L"]):
        ax.annotate(f"{v:,.0f}", (v, y), color=INK, fontsize=9, va="center",
                    ha="left" if v >= 0 else "right",
                    xytext=(4 if v >= 0 else -4, 0), textcoords="offset points")
    ax.set_xlim(lo - pad, hi + pad)   # keep zero in frame without wasting half the canvas
    ax.set_title("Net P&L by symbol (rupees)", color=INK, fontsize=13,
                 fontweight="bold", loc="left", pad=12)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    _style(ax)
    ax.grid(axis="y", visible=False)
    ax.grid(axis="x", color="#eeedea", linewidth=1)
    plt.tight_layout(); plt.show()

---
## Troubleshooting

| Symptom | Fix |
| --- | --- |
| A symbol is skipped with `no LTF data` | Usually the Yahoo limit: `15m` history only goes back ~60 days. Use a recent range, switch to `60m`/`1d`, or use the `csv` source. |
| `WARNING: Yahoo serves at most ~60 days...` | Exactly that — your start date is too far back for the interval. |
| `Not enough LTF data bars available` | The range is shorter than the 216-bar warmup. Widen the dates or lower `EMA_TP_PERIOD`. |
| `Interval ... not supported` | Use `5m`, `15m`, `30m`, `60m`, `1d` or `1mo`. |
| `CSV not found` | The `csv` source wants `<TICKER>_<interval>.csv`, e.g. `RELIANCE_15m.csv` **and** `RELIANCE_60m.csv` — one file per timeframe. |
| Backtest completes but zero trades | Normal with strict filters. Try a longer window, a lower `MIN_TRADE_SCORE`, a lower `MIN_WICK_TO_RANGE_RATIO`, or `MAX_ZONE_WIDTH_ATR_MULTIPLE = 999`. |
| Empty or odd prices | Yahoo data is unofficial and occasionally misprints. Spot-check anything surprising. |
| Throttling / repeated empty results | Raise `API_DELAY_SECONDS` in Step 5. |
| Results look doubled | Re-run the Step 6 cell — it clears previous results at the top. |

### Strategy reference

Full explanation of the zones, filters, scoring and entry types is in
[`gtf_strategy/README.md`](https://github.com/ersid111/Stocks/blob/main/gtf_strategy/README.md).